In [1]:
# Cell 1: Import library

import os
import numpy as np
import pandas as pd
from scipy import stats
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine, text
from pathlib import Path

In [3]:
# Cell 2: Connect to MySQL
PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")
user = os.getenv("MYSQL_USER")
pwd = quote_plus(os.getenv("MYSQL_PASSWORD"))
host = os.getenv("MYSQL_HOST")
port = os.getenv("MYSQL_PORT")
db = os.getenv("MYSQL_DB")
engine = create_engine(f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}")

with engine.connect() as conn:
    print("Current database:", conn.execute(text("SELECT DATABASE()")).scalar())

Current database: olist_ecommerce


In [5]:
# Cell 3: Delivered valid orders with their latest review
q = """
SELECT v.order_id, v.customer_state,
       v.order_purchase_timestamp, v.order_delivered_customer_date,
       v.order_estimated_delivery_date, r.review_score
FROM valid_orders_t v
LEFT JOIN latest_review_t r ON r.order_id = v.order_id
WHERE v.order_status = 'delivered' AND v.order_delivered_customer_date IS NOT NULL
"""

df = pd.read_sql(text(q), engine, parse_dates=["order_purchase_timestamp",
                "order_delivered_customer_date", "order_estimated_delivery_date"])

df["delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.total_seconds() / 86400
df["is_late"] = df["delay_days"] > 0
df["delay_bucket"] = pd.cut(df["delay_days"], bins = [-np.inf, -7, 0, 7, np.inf],
                            labels=["1. early 7+ d", "2. early 0-7 d", "3. late up to 7 d", "4. late 7+ d"])

print(df.shape)
df.head()

(96469, 9)


,order_id,customer_state,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,review_score,delay_days,is_late,delay_bucket
0,00010242fe8c5a6d1ba2dd792cb16214,RJ,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,5.0,-8.011250,False,1. early 7+ d
1,00018f77f2f0320c557190d7a144bdd3,SP,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,4.0,-2.330278,False,2. early 0-7 d
2,000229ec398224ef6ca0657da4fc703e,MG,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,5.0,-13.444954,False,1. early 7+ d
3,00024acbcdf0a6daa1e931b038114c75,SP,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,4.0,-5.435660,False,2. early 0-7 d
4,00042b26cf59d7ce69dfabb4e55b4fd9,SP,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,5.0,-15.303808,False,1. early 7+ d


In [6]:
# Cell 4: Checks against notebook 6
rev = df.dropna(subset=["review_score"])
checks = {
    "delivered orders = 96469":      len(df) == 96469,
    "late rate = 8.11%":             round(df["is_late"].mean() * 100, 2) == 8.11,
    "reviewed on time = 88163":      int((~rev["is_late"]).sum()) == 88163,
    "reviewed late = 7660":          int(rev["is_late"].sum()) == 7660,
    "avg review on time = 4.29":     round(rev.loc[~rev["is_late"], "review_score"].mean(), 2) == 4.29,
    "avg review late = 2.57":        round(rev.loc[rev["is_late"], "review_score"].mean(), 2) == 2.57,
}

for k, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", k)
print(rev["delay_bucket"].value_counts().sort_index())

PASS - delivered orders = 96469
PASS - late rate = 8.11%
PASS - reviewed on time = 88163
PASS - reviewed late = 7660
PASS - avg review on time = 4.29
PASS - avg review late = 2.57
delay_bucket
1. early 7+ d        70929
2. early 0-7 d       17234
3. late up to 7 d     4409
4. late 7+ d          3251
Name: count, dtype: int64


In [7]:
# Cell 5: Mann-Whitney U, late vs on-time review scores
ontime = rev.loc[~rev["is_late"], "review_score"].to_numpy()
late = rev.loc[rev["is_late"], "review_score"].to_numpy()

u, p = stats.mannwhitneyu(ontime, late, alternative="two-sided")
prob_sup = u / (len(ontime) * len(late))    # P(on-time reveiw > late reveiw), ties count half
rank_biserial = 2 * prob_sup - 1

rng = np.random.default_rng(42)
diffs = [rng.choice(ontime, len(ontime)).mean() - rng.choice(late, len(late)).mean() for _ in range(2000)]
ci = np.percentile(diffs, [2.5, 97.5])


print(f"mean on time {ontime.mean():.2f}, mean late {late.mean():.2f}, difference {ontime.mean() - late.mean():.2f}")
print(f"95% bootstrap CI of the difference: [{ci[0]:.2f}, {ci[1]:.2f}]")
print(f"Mann-Whitney U p-value: {p:.3g}")
print(f"P(random on-time review > random late review): {prob_sup:.3f}")
print(f"rank-biserial correlation: {rank_biserial:.3f}")

mean on time 4.29, mean late 2.57, difference 1.73
95% bootstrap CI of the difference: [1.69, 1.77]
Mann-Whitney U p-value: 0
P(random on-time review > random late review): 0.777
rank-biserial correlation: 0.554


In [8]:
# Cell 6: Kruskal-Wallis across delay buckets, then pariwise Mann-Whitney (Bonferroni)

from itertools import combinations
groups = {b: g["review_score"].to_numpy() for b, g in rev.groupby("delay_bucket", observed=True)}
H, p = stats.kruskal(*groups.values())
n = sum(len(g) for g in groups.values())
print(f"Kurskal-Wallis H = {H:.1f}, p = {p:.3g}, epsilon-squared = {H / (n - 1):.3f}")

pairs = list(combinations(groups, 2))
for a, b in pairs:
    u, pp = stats.mannwhitneyu(groups[a], groups[b], alternative="two-sided")
    print(f"{a} vs {b}: p (Bonferroni) = {min(pp * len(pairs), 1):.3g}, "
          f"means {groups[a].mean():.2f} vs {groups[b].mean():.2f}")

Kurskal-Wallis H = 9665.3, p = 0, epsilon-squared = 0.101
1. early 7+ d vs 2. early 0-7 d: p (Bonferroni) = 2.84e-49, means 4.32 vs 4.20
1. early 7+ d vs 3. late up to 7 d: p (Bonferroni) = 0, means 4.32 vs 3.18
1. early 7+ d vs 4. late 7+ d: p (Bonferroni) = 0, means 4.32 vs 1.73
2. early 0-7 d vs 3. late up to 7 d: p (Bonferroni) = 0, means 4.20 vs 3.18
2. early 0-7 d vs 4. late 7+ d: p (Bonferroni) = 0, means 4.20 vs 1.73
3. late up to 7 d vs 4. late 7+ d: p (Bonferroni) = 2.91e-309, means 3.18 vs 1.73


In [10]:
# Cell 7: Chi-square on late vs not late by customer state (states with 100+ orders), plus SP vs the rest
counts = df.groupby("customer_state")["is_late"].agg(["size", "sum"])
counts = counts[counts["size"] >= 100]
table = np.column_stack([counts["sum"], counts["size"] - counts["sum"]])
chi2, p, dof, _ = stats.chi2_contingency(table)
cramers_v = np.sqrt(chi2 / table.sum())    # 2 columns, so min(r, c) -1 = 1
print(f"{len(counts)} states: chi2 = {chi2:.0f}, dof = {dof}, p = {p:.3g}, Cramer's V = {cramers_v:.3f}")

sp = df["customer_state"] == "SP"
t2 = pd.crosstab(sp, df["is_late"]).to_numpy()
chi2_sp, p_sp, _, _ = stats.chi2_contingency(t2, correction=False)
odds = (t2[0, 1] / t2[0, 0]) / (t2[1, 1] / t2[1, 0])   # non-SP odds of late / SP odds of late
print(f"SP late rate {df.loc[sp, 'is_late'].mean() * 100:.2f}% vs rest {df.loc[~sp, 'is_late'].mean() * 100:.2f}%")
print(f"SP vs rest: p = {p_sp:.3g}, odds ratio (rest vs SP) = {odds:.2f}")

24 states: chi2 = 1619, dof = 23, p = 0, Cramer's V = 0.130
SP late rate 5.89% vs rest 9.72%
SP vs rest: p = 2.86e-102, odds ratio (rest vs SP) = 1.72


## Statistical findings

**Data:** 96,469 delivered valid orders (95,823 with a review). All reconciliation checks against notebook 6 pass. Review scores are 1 to 5 ratings, so rank-based tests are used (Mann-Whitney U, Kruskal-Wallis). With this many orders, p-values are almost always tiny, so effect sizes are reported next to them.

### 1. Late delivery and review score
- On-time orders average 4.29 stars and late orders 2.57, a difference of 1.73 (95% bootstrap CI 1.69 to 1.77).
- Mann-Whitney U: p < 0.001. A random on-time review is higher than a random late review in 77.7% of pairs; the rank-biserial correlation is 0.55, a large effect.

### 2. Size of the delay
- Kruskal-Wallis across four delay buckets: H = 9,665, p < 0.001, epsilon-squared = 0.10 (medium effect).
- All pairwise comparisons are significant after Bonferroni correction (6 comparisons).
- Mean scores: early 7+ days 4.32, early 0 to 7 days 4.20, late up to 7 days 3.18, late more than 7 days 1.73. Being slightly earlier or later within the early range matters little (0.12 stars); the main drops come when an order becomes late and when it is more than 7 days late.

### 3. Late rate by customer state
- The late rate differs across the 24 states with 100+ orders: chi-square = 1,619, dof = 23, p < 0.001, Cramer's V = 0.13 (small effect).
- SP is late 5.89% of the time against 9.72% for all other states; the odds of a late delivery are 1.72 times higher outside SP (p < 0.001).

### Limits
- These are observational results. They show that late delivery goes with low scores, not that it causes them.
- Only orders with a review are used in tests 1 and 2. Customers with a bad experience may be more likely to leave a review, so the true gap could differ.
- A review score reflects more than delivery (product quality, price), so some of the gap may come from other factors that correlate with lateness.
- The state result does not explain why states differ; distance, carriers and seller location are not tested here.

In [11]:
def fmt_p(p):
    return "< 0.001" if p < 0.001 else f"= {p:.3g}"

In [14]:
print(f"Mann-Whitney U p-value: {fmt_p(p)}")

Mann-Whitney U p-value: < 0.001
